# Exp 3 · AnyUp3D diagnosis at a usable resolution (T12)

**Question:** why AnyUp3D falls below AnyUp2D once the resolution is usable. T11 run 5 (DINOv2-B 448² → 112², 5 clips): AnyUp2D 0.763 vs AnyUp3D 0.562, and AnyUp3D still loses 11 % of object-frames. Here AnyUp2D is the only reference (no bilinear). Clips: **parkour** (collapse), **bmx-trees** (thin object), **blackswan** (easy control).

**What the checkpoint saw in training** (`Train_3D_full.ipynb`; Cell 3 prints the checkpoint's own `t_schedule`):
- **1 feature frame** per call up to step 5000, then **2** (step 5000–6500). The T = 8 stage starts at step 15000, after this checkpoint.
- VideoMAE features, guide with **2 frames per feature frame**, 7² → 14² upsampling.

At inference we give it 8 DINOv2 frames per call, one guide frame per feature frame.

| Cell | Run | Backbone | Upsampler | Frames per AnyUp3D call | What it tests |
|---|---|---|---|---|---|
| 4 | `ref2d` | DINOv2-B 448² | AnyUp2D | — | reference (T11 run 5) |
| 4 | `t8` | DINOv2-B 448² | AnyUp3D | 8, `window_t 4` | reference (T11 run 5); checked per clip |
| 5 | `t1` | DINOv2-B 448² | AnyUp3D | **1** | no temporal partner at all: spatial path only |
| 5 | `t2` | DINOv2-B 448² | AnyUp3D | **2** | context length as at the end of training |
| 5 | `t8_wt0` | DINOv2-B 448² | AnyUp3D | 8, **`window_t 0`** | temporal attention off; temporal convs and time-RoPE still on |
| 6 | `v_2d` | VideoMAE 224 | AnyUp2D | — | reference on AnyUp3D's training backbone |
| 6 | `v_3d` | VideoMAE 224 | AnyUp3D | 8 (16 guide frames, **2:1 as in training**) | AnyUp3D in distribution (backbone, guide ratio) |
| 6b | `v448_2d` | VideoMAE **448** (28² tokens) | AnyUp2D | — | reference on VideoMAE at a usable resolution |
| 6b | `v448_3d` | VideoMAE 448 | AnyUp3D | 8 (guide 2:1) | **main run 2 question:** AnyUp3D vs AnyUp2D on its own backbone, usable resolution |
| 6b | `v448_t1` | VideoMAE 448 | AnyUp3D | **1** | temporal context removed |
| 6b | `v448_wt0` | VideoMAE 448 | AnyUp3D | 8, **`window_t 0`** | temporal attention off |

Propagation grid is 112² everywhere, with the Exp 3 rule (τ 0.07, top-7, frame 0 + last 6 frames) and official DAVIS J&F. VideoMAE has one feature frame per 2 video frames, so propagation runs on the feature frames, and feature frame k predicts video frames 2k and 2k+1 (same rule for both upsamplers).

**Feature diagnostics** (every run, no J&F needed): per frame, cos(AnyUp3D, AnyUp2D) on the same backbone chunk, and the **input consistency** of each upsampler, i.e. its output avg-pooled back to the LR grid vs the LR features (1 = it keeps what the backbone said).

**How to read it**
- `t1` ≈ `ref2d` → the temporal part hurts. Then `t2` vs `t8` (context length) and `t8_wt0` vs `t8` (temporal attention) say which part.
- `t1` ≈ `t8` (still bad) → the spatial path / backbone domain. Then Test V: `v_3d` ≥ `v_2d` means AnyUp3D works on its own backbone (backbone shift); `v_3d` < `v_2d` means it is weak even in distribution.
- Low input consistency for AnyUp3D only → it rewrites the features instead of upsampling them.

**Run 2 (VideoMAE at 448):** run 1 showed VideoMAE 224 has only 14² tokens, where every upsampler fails (as in Problem 1). Cell 6b runs VideoMAE at 448 → 28² tokens; videomae-base was trained at 224, so its fixed position table is interpolated bicubically per feature frame (zero-shot). Cell 6b first prints a check: VideoMAE 448 features pooled to 14² vs its 224 features, with DINOv2 448 vs 224 as the reference. If VideoMAE is far below DINOv2 there, the backbone itself breaks at 448 and the 6b numbers must be read with that in mind. The flags `RUN_REF`, `RUN_T`, `RUN_V224`, `RUN_V448` in Cell 2 choose the tests (default: only 6 and 6b, ≈ 15–20 min, estimate). Outputs go to `n3_<RUN_TAG>`, and Cell 7 lists the run 1 results (marked *) for the runs that were not repeated.

**Run:** T4 runtime, then Run all (≈ 30–45 min, estimate). **Runtime → Restart session** first if `anyup` was already imported. Cells 4, 5 and 6 are independent after Cell 3, and Cell 7 shows whatever finished, also after an interrupt (Runtime → Interrupt, not restart). A run that runs out of GPU memory is reported as SKIPPED and the others continue.

In [ ]:
# ── Cell 1 · Repo + Drive + deps ────────────────────────────────────────────
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"   # deterministic cuBLAS (T07); must be set before anything starts CUDA
import sys, subprocess
from google.colab import drive
drive.mount("/content/drive")

REPO_URL    = "https://github.com/mu-az88/anyup.git"
REPO_BRANCH = "experiments-in-action-test"   # T12 (same branch as T10 / T11)
REPO_DIR    = "/content/anyup"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    # existing clone may be on another branch — switch to REPO_BRANCH and update it
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"], check=True)
if "anyup" in sys.modules:
    raise RuntimeError("anyup was already imported in this runtime — Runtime → Restart session, then run again")
subprocess.run(["pip", "install", "-e", REPO_DIR, "-q"], check=True)
subprocess.run(["pip", "install", "transformers", "timm", "scikit-image", "scipy", "imageio-ffmpeg", "-q"], check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# ── Cell 2 · Config ─────────────────────────────────────────────────────────
CLIPS        = ["parkour", "bmx-trees", "blackswan"]   # ↓ clips (time ∝ their frames: 100 + 80 + 50). Collapse, thin object, easy control
OUTPUT_ROOT  = "/content/drive/MyDrive/results/T12/anyup3d_diag"
RUN_TAG      = "v448"       # outputs go to OUTPUT_ROOT/n3_<RUN_TAG> ("" = n3, the folder of run 1; do not overwrite it)
EARLIER_DIR  = OUTPUT_ROOT + "/n3"   # run 1 results: Cell 7 lists runs found there that were not re-run now (marked *)

# ── which tests to run (a cell with False skips itself; Cell 7 shows whatever finished + the earlier ones) ──
RUN_REF      = False        # Cell 4:  ref2d, t8            (DINOv2; done in run 1, ≈ 7 min)
RUN_T        = False        # Cell 5:  t1, t2, t8_wt0       (DINOv2; done in run 1, ≈ 11 min)
RUN_V224     = True         # Cell 6:  v_2d, v_3d           (VideoMAE 224, 14² tokens; ≈ 3 min; same-session reference for 6b)
RUN_V448     = True         # Cell 6b: v448_2d, v448_3d, v448_t1, v448_wt0 + the VideoMAE 448-vs-224 feature check
DAVIS_DEST   = "/content"
DEVICE       = "cuda"
SEED         = 0
CKPT_ANYUP3D = "/content/drive/MyDrive/results/anyup3d_step6500.pth"   # as Exp 3 / T11 (checkpoint choice S06 open)

# DINOv2 (Cells 4–5)
DINO_INPUT   = (448, 448)   # DINOv2-B input (squashed, = T11 e448_p112) → 32² tokens. Fixed by the Q&A (no resolution sweep).
                            #   Dependent: BB_CHUNK (DINOv2 attention ∝ tokens²), the T11 check values in Cell 4
GRID         = (112, 112)   # propagation grid = upsampler output = guide size. Clip features on GPU = T·12544·768·4 B (38.5 MB per
                            #   frame: 3.9 GB for parkour). Dependent: Q_CHUNK_3D, Q_CHUNK_PROP, frames per call in RUNS
BB_CHUNK     = 8            # ↓ frames per DINOv2 call. AnyUp3D sub-calls split each backbone chunk, so the largest `fpc` in RUNS
                            #   must divide it (8 = T11 run 5: AnyUp3D peak 6.5 GiB per call + 3.9 GB store on a T4). Dependent: RUNS fpc

# VideoMAE (Cells 6, 6b): 16 frames → 8 feature frames (tubelet 2); patch 16 → 14² tokens at 224, 28² at 448
VMAE_CHUNK   = 16           # frames per VideoMAE call (must be 16 for videomae-base). AnyUp3D gets up to 8 feature frames + 16 guide frames
                            #   of a chunk per call (fpc 8 mask: (8·12544) × (8·h·w) bools ≈ 0.16 GB at 14², 0.63 GB at 28²). Dependent: RUNS v* fpc
VMAE_HI      = (448, 448)   # ↓ VideoMAE input for Cell 6b → 28² tokens (DINOv2 448 has 32²; (512, 512) gives 32²). Not native: the fixed
                            #   position table is interpolated (anyup.backbones.interpolate_videomae_pos; zero-shot). Backbone attention ∝
                            #   (8·h·w)²: 6272 tokens at 448 ≈ 1.9 GB per layer in eager mode. Dependent: Q_CHUNK_3D, RUNS v448_* fpc

# Upsamplers
Q_CHUNK_3D   = 1024         # ↓ AnyUp3D query pixels per attention call; memory ≈ Q_CHUNK_3D × fpc·h·w (LR) × 4 heads × 4 B (+ copies).
                            #   The full bool mask (fpc·H·W) × (fpc·h·w) is built anyway (Problem 3). Dependent: GRID, RUNS fpc
RUNS = {   # name → backbone, upsampler, AnyUp3D frames per call (fpc), window_t (training value 4; not stored in the checkpoint)
    "ref2d":  dict(bb="dinov2",   cond="anyup2d"),
    "t8":     dict(bb="dinov2",   cond="anyup3d", fpc=8, window_t=4),   # ↓ fpc 8: one call peaks at 6.5 GiB (T11 memory test, T4)
    "t1":     dict(bb="dinov2",   cond="anyup3d", fpc=1, window_t=4),   # 1 frame per call (window_t has no effect at T = 1)
    "t2":     dict(bb="dinov2",   cond="anyup3d", fpc=2, window_t=4),
    "t8_wt0": dict(bb="dinov2",   cond="anyup3d", fpc=8, window_t=0),   # attention to the own frame only; same memory as t8
    "v_2d":   dict(bb="videomae", cond="anyup2d", inp=224),
    "v_3d":   dict(bb="videomae", cond="anyup3d", inp=224, fpc=8, window_t=4),   # fpc must divide 8 (= VMAE_CHUNK / 2)
    "v448_2d":  dict(bb="videomae", cond="anyup2d", inp=VMAE_HI),
    "v448_3d":  dict(bb="videomae", cond="anyup3d", inp=VMAE_HI, fpc=8, window_t=4),   # ↓ fpc 8: mask 0.63 GB, est. peak ≈ t8 (10.4 GiB)
    "v448_t1":  dict(bb="videomae", cond="anyup3d", inp=VMAE_HI, fpc=1, window_t=4),   # 1 feature frame (+ its 2 guide frames) per call
    "v448_wt0": dict(bb="videomae", cond="anyup3d", inp=VMAE_HI, fpc=8, window_t=0),   # temporal attention off; same memory as v448_3d
}
CELL4_RUNS, CELL5_RUNS, CELL6_RUNS = ["ref2d", "t8"], ["t1", "t2", "t8_wt0"], ["v_2d", "v_3d"]
CELL6B_RUNS  = ["v448_2d", "v448_3d", "v448_t1", "v448_wt0"]
T11_RUN5 = {"ref2d": {"blackswan": 0.937, "bmx-trees": 0.561, "parkour": 0.880},   # clip-mean J&F, T11 run 5 printout
            "t8":    {"blackswan": 0.880, "bmx-trees": 0.327, "parkour": 0.525}}   # (exp3_debug_E.ipynb, e448_p112, commit 9c34ff5)
CHECK_TOL    = 0.002        # |J&F − T11 run 5| per clip for PASS (the printout is rounded to 3 decimals)

# Propagation (Exp 3 rule)
TAU, TOP_K   = 0.07, 7
N_LAST       = 6            # ↓ context frames besides frame 0. VideoMAE: counted in FEATURE frames (6 ≈ 12 video frames) so both backbones
                            #   use the same number of context feature maps. Dependent: Q_CHUNK_PROP (block width = (1+N_LAST)·H·W)
Q_CHUNK_PROP = 1024         # ↓ query pixels per propagation block; block = Q_CHUNK_PROP × (1+N_LAST)·H·W floats (112²: ≈ 360 MB).
                            #   Dependent: GRID, N_LAST

# Visuals (display only)
SHOW_FRACS   = [0.5, 0.9]   # mask figure: frames at these fractions of each clip, plus the first frame where t8 loses an object
SHOW_H       = 150          # panel height in px (width keeps the aspect ratio)
VIDEO_FPS    = 8
PCA_SAMPLE   = 20000        # tokens per run for the PCA fit (speed only)

In [ ]:
# ── Cell 3 · Imports, data, models, helpers ─────────────────────────────────
import time, json, gc
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt

from anyup import davis, metrics, viz
from anyup.backbones import BackboneLoader, BackboneOutput, resize_frames, normalize_frames, IMAGENET_MEAN, IMAGENET_STD
from anyup.video import iter_backbone_chunks
from anyup.upsamplers import UpsamplerLoader, load_anyup3d_checkpoint
from anyup.modules import attention_masking3d
from anyup.utils.seed import set_seed

set_seed(SEED)
CUDA = torch.cuda.is_available()
GPU_NAME = torch.cuda.get_device_name(0) if CUDA else "cpu"
print("GPU:", GPU_NAME)
ROOT = davis.download_davis(DAVIS_DEST)
missing = [c for c in CLIPS if c not in davis.list_clips(ROOT, "val")]
assert not missing, f"not DAVIS-2017 val clips: {missing}"
OUT_DIR = Path(OUTPUT_ROOT) / (f"n{len(CLIPS)}" + (f"_{RUN_TAG}" if RUN_TAG else ""))
OUT_DIR.mkdir(parents=True, exist_ok=True)
NFR = {c: len(davis.frame_paths(ROOT, c)) for c in CLIPS}
print("clips:", ", ".join(f"{c} {n} frames" for c, n in NFR.items()), "| outputs →", OUT_DIR)
assert all(BB_CHUNK % RUNS[r]["fpc"] == 0 for r in RUNS if RUNS[r]["bb"] == "dinov2" and RUNS[r]["cond"] == "anyup3d"), \
    "every DINOv2 fpc must divide BB_CHUNK"
assert all((VMAE_CHUNK // 2) % RUNS[r]["fpc"] == 0 for r in RUNS if RUNS[r]["bb"] == "videomae" and RUNS[r]["cond"] == "anyup3d"), \
    "every VideoMAE fpc must divide VMAE_CHUNK / 2"

# ── what the checkpoint was trained with (facts behind the T12 tests) ──
_ck = load_anyup3d_checkpoint(CKPT_ANYUP3D)
_cfg = _ck.get("cfg", {})
CKPT_INFO = dict(global_step=_ck.get("global_step"),
                 t_schedule=[dict(t=s.t, start_step=s.start_step, batch_size=s.batch_size) for s in _cfg.get("t_schedule", [])],
                 **{k: _cfg.get(k) for k in ("encoder_model", "crop_size", "img_size", "window_t", "window_ratio", "t_k")})
del _ck, _cfg
_seen = [s for s in CKPT_INFO["t_schedule"] if CKPT_INFO["global_step"] is not None and s["start_step"] <= CKPT_INFO["global_step"]]
print(f"checkpoint: step {CKPT_INFO['global_step']}, t_schedule {CKPT_INFO['t_schedule']}")
print(f"  stages reached: video T = {[s['t'] for s in _seen]} → feature frames per call (tubelet 2) = {[max(1, s['t'] // 2) for s in _seen]}")
print(f"  crop {CKPT_INFO['crop_size']} (guide + GT) / img {CKPT_INFO['img_size']} (LR input) → LR 7² → 14²; window_t {CKPT_INFO['window_t']}")

# ── models (loaded once; AnyUp3D per window_t on first use) ──
UP2D = UpsamplerLoader.load("anyup2d", device=DEVICE)
_UP3D, _BB = {}, {}

def get_up3d(window_t):
    if window_t not in _UP3D:
        _UP3D[window_t] = UpsamplerLoader.load("anyup3d", device=DEVICE, ckpt_path=CKPT_ANYUP3D,
                                               q_chunk_size=Q_CHUNK_3D, window_t=window_t)
    return _UP3D[window_t]

def get_bb(name, inp=224):
    """DINOv2 at DINO_INPUT, or VideoMAE at `inp` (224 native; other sizes with the interpolated position table)."""
    key = (name, inp if name == "videomae" else DINO_INPUT)
    if key not in _BB:
        _BB[key] = (BackboneLoader.load("dinov2", device=DEVICE, input_size=DINO_INPUT, guide_size=GRID, variant="base")
                    if name == "dinov2" else
                    BackboneLoader.load("videomae", device=DEVICE, input_size=inp, guide_size=GRID, interpolate_pos=True))
    return _BB[key]

# results: run → clip → ...
RES, PRED, DIAG, LOST, PCA_FEAT, RUNINFO, SKIPPED = {}, {}, {}, {}, {}, {}, {}
PCA_FRAME = {c: 2 * ((NFR[c] // 2) // 2) for c in CLIPS}   # middle frame, even so VideoMAE has its own feature frame there


def clip_mean_jf(res):
    return (float(np.mean(res["J_M"])) + float(np.mean(res["F_M"]))) / 2


def propagate_e(feats_flat, hw, ann0, tau, top_k, n_last, q_chunk=Q_CHUNK_PROP):
    """
    T11 propagate_e without the window (checked there = davis.propagate_labels, 0 differing pixels): each query pixel keeps its
    top_k context matches, softmax over them with temperature tau, label = weighted mean of their soft labels.
    feats_flat (T, N, C) unit vectors on GPU; hw = (h, w); ann0 (H0, W0) GT of frame 0. → soft labels (T, n_ch, h, w).
    """
    T, N, C = feats_flat.shape
    h, w = hw
    n_ch = int(ann0.max()) + 1
    labels = torch.zeros(T, n_ch, N, device=feats_flat.device)                          # (T, n_ch, N)
    labels[0] = davis._label_maps_frame0(ann0, n_ch - 1, h, w).to(feats_flat.device)
    for t in range(1, T):
        ctx = [0] + list(range(max(1, t - n_last), t))                                   # frame 0 always first
        k_feats = feats_flat[ctx].reshape(-1, C)                                         # (Nctx·N, C), frame-major
        k_labels = labels[ctx].permute(1, 0, 2).reshape(n_ch, -1)                        # (n_ch, Nctx·N)
        for q0 in range(0, N, q_chunk):
            q1 = min(N, q0 + q_chunk)
            A = feats_flat[t, q0:q1] @ k_feats.T / tau                                   # (Q, Nctx·N)
            vals, idx = A.topk(top_k, dim=-1)                                            # (Q, top_k)
            wts = F.softmax(vals, dim=-1)                                                # (Q, top_k)
            labels[t, :, q0:q1] = (k_labels[:, idx] * wts[None]).sum(-1)                 # (n_ch, Q, top_k) → (n_ch, Q)
            del A
    return labels.view(T, n_ch, h, w)


@torch.no_grad()
def frame_cos(a, b):
    """(K, C, H, W) × 2 → list of K floats: mean over pixels of the per-pixel cosine (float64)."""
    return [float(F.cosine_similarity(a[k].double(), b[k].double(), dim=0).mean()) for k in range(a.shape[0])]


@torch.no_grad()
def up3d_split(up3d, ch, fpc):
    """AnyUp3D on sub-calls of fpc consecutive frames of one backbone chunk → (K, C, H, W), position of each frame in its call."""
    outs, pos = [], []
    for s in range(0, ch.features.shape[0], fpc):
        sub = BackboneOutput(features=ch.features[s:s + fpc], guide=ch.guide[s:s + fpc], meta=ch.meta)   # (≤fpc, C, h, w), (≤fpc, 3, H, W)
        o = up3d(sub, output_size=GRID).features                                        # (≤fpc, C, H, W)
        outs.append(o); pos += list(range(o.shape[0]))
    return torch.cat(outs), pos


@torch.no_grad()
def up3d_vmae(up3d, lr, fidx, src, n_frames):
    """
    AnyUp3D on one VideoMAE chunk as in training: guide = BOTH video frames of each tubelet (f, f+1; the last frame repeats at the
    clip end), so 2K guide frames for K feature frames (AnyUp3D pools the guide encoding to K in time).
    lr (K, C, h, w) raw VideoMAE features; fidx = video frame of each feature frame. → (K, C, H, W).
    """
    gidx = [g for f in fidx for g in (f, min(f + 1, n_frames - 1))]                     # 2K video frames
    frames = src.get(gidx).to(DEVICE, torch.float32)                                     # (2K, 3, 480, W) [0, 1]
    guide = normalize_frames(resize_frames(frames, GRID), IMAGENET_MEAN, IMAGENET_STD)  # (2K, 3, H, W), as the adapter's guide
    out = up3d.model(guide.permute(1, 0, 2, 3).unsqueeze(0),                             # (1, 3, 2K, H, W)  guide first
                     lr.float().permute(1, 0, 2, 3).unsqueeze(0),                        # (1, C, K, h, w)
                     output_size=GRID, q_chunk_size=Q_CHUNK_3D)                          # → (1, C, K, H, W)
    return out[0].permute(1, 0, 2, 3).float()                                            # (K, C, H, W)


def clip_features(spec, clip):
    """
    Backbone → AnyUp2D (always: reference for the diagnostics) → the run's upsampler → L2-normalized features of the run in one
    (K, N, C) GPU tensor (K = feature frames). Returns flat, per-frame diagnostics, the PCA frame (CPU, fp16).
    """
    vmae = spec["bb"] == "videomae"
    bb, src = get_bb(spec["bb"], spec.get("inp", 224)), davis.open_clip(ROOT, clip)
    T = len(src)
    up3d = get_up3d(spec["window_t"]) if spec["cond"] == "anyup3d" else None
    d = dict(frame=[], pos=[], cos_2d=[], ic_up=[], ic_2d=[])
    flat, k0, keep = None, 0, None
    for ch in iter_backbone_chunks(bb, src, chunk_size=VMAE_CHUNK if vmae else BB_CHUNK, overlap=0):
        lr, fidx = ch.features, ch.meta["frame_index"]                                   # (K, C, h, w) raw; video frame per feature frame
        f2 = UP2D(ch, output_size=GRID).features                                         # (K, C, H, W)
        if spec["cond"] == "anyup2d":
            f, pos = f2, [0] * len(fidx)
        elif vmae:                                                                       # sub-calls of fpc feature frames
            subs = range(0, len(fidx), spec["fpc"])
            f = torch.cat([up3d_vmae(up3d, lr[s:s + spec["fpc"]], fidx[s:s + spec["fpc"]], src, T) for s in subs])   # (K, C, H, W)
            pos = [p for s in subs for p in range(len(fidx[s:s + spec["fpc"]]))]
        else:
            f, pos = up3d_split(up3d, ch, spec["fpc"])
        lr_hw = tuple(lr.shape[-2:])
        d["frame"] += list(fidx); d["pos"] += pos
        d["cos_2d"] += frame_cos(f, f2)
        d["ic_up"] += frame_cos(F.adaptive_avg_pool2d(f, lr_hw), lr)                     # output pooled back to the LR grid vs LR
        d["ic_2d"] += frame_cos(F.adaptive_avg_pool2d(f2, lr_hw), lr)
        K, C, H, W = f.shape
        if flat is None:
            flat = torch.empty(len(range(0, T, 2)) if vmae else T, H * W, C, device=f.device)   # (K_total, N, C)
        flat[k0:k0 + K] = F.normalize(f.reshape(K, C, H * W).permute(0, 2, 1), dim=-1)  # (K, N, C) unit vectors
        if PCA_FRAME[clip] in fidx:
            j = fidx.index(PCA_FRAME[clip])
            keep = dict(up=f[j].half().cpu(), lr=lr[j].half().cpu())                     # (C, H, W), (C, h, w)
        k0 += K
        del f, f2, lr
    assert k0 == flat.shape[0], f"feature frames {k0} != {flat.shape[0]}"
    if vmae:
        assert d["frame"] == list(range(0, T, 2)), "VideoMAE frame_index is not 0, 2, 4, …"
    return flat, d, keep


def to_native(pred_lr, gt):
    return np.stack([davis.resize_labels(np.asarray(p, np.uint8), gt.shape[1:]) for p in pred_lr])   # (T, H0, W0), nearest


def lost_frames(gt, nat):
    """(T, n_obj) bool present / lost: object in the GT of frame t, and 0 predicted pixels of it (lost)."""
    n_obj = int(gt[0][gt[0] != 255].max())
    present = np.stack([[(g == o).any() for o in range(1, n_obj + 1)] for g in gt])
    hit = np.stack([[(p == o).any() for o in range(1, n_obj + 1)] for p in nat])
    return present, present & ~hit


def run(name):
    """One run over all clips: features → propagation (Exp 3 rule) on the 112² grid → official J&F + diagnostics."""
    spec = RUNS[name]
    if name in RES and len(RES[name]) == len(CLIPS):
        print(f"{name}: already done, skipped (delete RES['{name}'] to re-run)"); return
    for dct in (RES, PRED, DIAG, LOST, PCA_FEAT):
        dct[name] = {}
    if CUDA:
        torch.cuda.reset_peak_memory_stats()
    t_run = time.perf_counter()
    for ci, clip in enumerate(CLIPS):
        t0 = time.perf_counter()
        gt = davis.load_annotations(ROOT, clip)                                          # (T, 480, W) uint8
        flat = soft = None
        try:
            flat, d, keep = clip_features(spec, clip)
            soft = propagate_e(flat, GRID, gt[0], TAU, TOP_K, N_LAST)                    # (K, n_ch, H, W)
            pred = soft.argmax(1).to(torch.uint8).cpu().numpy()                          # (K, H, W)
        except torch.cuda.OutOfMemoryError as e:
            SKIPPED[name] = f"CUDA out of memory on {GPU_NAME} at {clip}: {str(e).splitlines()[0][:150]}"
            print(f"SKIPPED {name}: {SKIPPED[name]}")
            break
        finally:
            del flat, soft
            gc.collect()
            if CUDA:
                torch.cuda.empty_cache()
        if spec["bb"] == "videomae":
            pred = pred[np.arange(len(gt)) // 2]                                         # hold: feature frame k → video frames 2k, 2k+1
        nat = to_native(pred, gt)
        RES[name][clip] = metrics.eval_clip_jf(gt, nat)
        PRED[name][clip] = pred
        DIAG[name][clip] = d
        LOST[name][clip] = lost_frames(gt, nat)
        PCA_FEAT[name][clip] = keep
        pr, lo = LOST[name][clip]
        print(f"[{ci + 1}/{len(CLIPS)}] {name:<8} {clip:<10} J&F {clip_mean_jf(RES[name][clip]):.3f} | lost {int(lo[1:-1].sum())}/{int(pr[1:-1].sum())}"
              + (f" | cos(3D,2D) {np.mean(d['cos_2d']):.3f}" if spec["cond"] == "anyup3d" else "") + f" | in-consistency up {np.mean(d['ic_up']):.3f} / 2D {np.mean(d['ic_2d']):.3f}"
              f" | {time.perf_counter() - t0:.0f}s")
    RUNINFO[name] = dict(seconds=round(time.perf_counter() - t_run, 1),
                         peak_gib=round(torch.cuda.max_memory_allocated() / 2**30, 2) if CUDA else None)
    attention_masking3d.compute_attention_mask_3d.cache_clear(); attention_masking3d.compute_attention_mask.cache_clear()
    gc.collect()
    if CUDA:
        torch.cuda.empty_cache()
    print(f"{name}: {RUNINFO[name]['seconds']:.0f}s, peak GPU {RUNINFO[name]['peak_gib']} GiB\n")


def summary_line(name):
    s = metrics.summarize_jf(RES[name])
    lost = sum(int(LOST[name][c][1][1:-1].sum()) for c in RES[name]); pres = sum(int(LOST[name][c][0][1:-1].sum()) for c in RES[name])
    return s, lost, pres

In [ ]:
# ── Cell 4 · Reference: AnyUp2D and AnyUp3D (8 per call) on DINOv2-B 448² → 112² ──
for name in (CELL4_RUNS if RUN_REF else []):
    run(name)
if not RUN_REF:
    print("Cell 4 skipped (RUN_REF = False)")
for name in CELL4_RUNS:   # check: same setting as T11 run 5 → same per-clip J&F
    for clip, ref in T11_RUN5.get(name, {}).items():
        if clip in RES.get(name, {}):
            v = clip_mean_jf(RES[name][clip])
            print(f"check {name}/{clip}: {v:.4f} vs T11 run 5 {ref:.3f} → {'PASS' if abs(v - ref) <= CHECK_TOL else 'WARN'}")

In [ ]:
# ── Cell 5 · Test T: temporal context and temporal attention (DINOv2-B 448² → 112²) ──
for name in (CELL5_RUNS if RUN_T else []):
    run(name)
if not RUN_T:
    print("Cell 5 skipped (RUN_T = False)")

In [ ]:
# ── Cell 6 · Test V: VideoMAE (AnyUp3D's training backbone, 224 native, guide 2:1) → 112² ──
for name in (CELL6_RUNS if RUN_V224 else []):
    run(name)
if not RUN_V224:
    print("Cell 6 skipped (RUN_V224 = False)")

In [ ]:
# ── Cell 6b · Test V448: VideoMAE at 448 (28² tokens, interpolated position table, guide 2:1) → 112² ──
@torch.no_grad()
def res_check(name, hi, lo, n_frames):
    """Does the backbone keep its features at the higher input? First chunk of each clip: features at `hi` avg-pooled to the grid at `lo`
    vs the features at `lo`, per-pixel cosine (float64). DINOv2 (trained up to 518) is the reference for a backbone that handles it."""
    out = {}
    for clip in CLIPS:
        x = davis.open_clip(ROOT, clip).get(list(range(n_frames)))                      # (n, 3, 480, W) [0, 1]
        a = hi(x).features; b = lo(x).features                                           # (K, C, h_hi, w_hi), (K, C, h_lo, w_lo)
        out[clip] = float(np.mean(frame_cos(F.adaptive_avg_pool2d(a, b.shape[-2:]), b)))
    print(f"  {name}: cos(pool(features at hi) → lo grid, features at lo) =", ", ".join(f"{c} {v:.3f}" for c, v in out.items()),
          f"| mean {np.mean(list(out.values())):.3f}")
    return out

if RUN_V448:
    print("Feature check, VideoMAE 448 vs 224 (zero-shot resolution change); DINOv2 448 vs 224 as reference:")
    RES_CHECK = dict(videomae=res_check("VideoMAE 448 vs 224", get_bb("videomae", VMAE_HI), get_bb("videomae", 224), VMAE_CHUNK))
    _d224 = BackboneLoader.load("dinov2", device=DEVICE, input_size=(224, 224), guide_size=GRID, variant="base")
    RES_CHECK["dinov2"] = res_check("DINOv2   448 vs 224", get_bb("dinov2"), _d224, BB_CHUNK)
    del _d224; gc.collect(); torch.cuda.empty_cache() if CUDA else None
    print()
    for name in CELL6B_RUNS:
        run(name)
else:
    print("Cell 6b skipped (RUN_V448 = False)")

In [ ]:
# ── Cell 7 · Results + visuals (works on whatever finished) ─────────────────
import sys
sys.last_traceback = sys.last_value = sys.last_type = None   # after an interrupt the traceback still holds GPU tensors
gc.collect()
if CUDA:
    torch.cuda.empty_cache()

done = [n for n in RUNS if RES.get(n) and len(RES[n]) == len(CLIPS)]
partial = [n for n in RUNS if RES.get(n) and n not in done]
GROUP = {n: (s["bb"], str(s.get("inp", DINO_INPUT))) for n, s in RUNS.items()}                 # backbone + input size
REF_OF = {n: next(m for m in RUNS if GROUP[m] == GROUP[n] and RUNS[m]["cond"] == "anyup2d") for n in RUNS}
COL = {"ref2d": "tab:orange", "t8": "tab:green", "t1": "tab:blue", "t2": "tab:cyan", "t8_wt0": "tab:olive",
       "v_2d": "tab:red", "v_3d": "tab:purple", "v448_2d": "tab:brown", "v448_3d": "tab:pink", "v448_t1": "tab:gray",
       "v448_wt0": "black"}
_inp = lambda s: "" if s["bb"] == "dinov2" else f" {s['inp'] if isinstance(s['inp'], int) else s['inp'][0]}"
DESC = {n: (f"{'DINOv2 448' if s['bb'] == 'dinov2' else 'VideoMAE'}{_inp(s)} · "
            + ("AnyUp2D" if s["cond"] == "anyup2d" else f"AnyUp3D {s['fpc']}/call wt{s['window_t']}")) for n, s in RUNS.items()}
print(f"finished on all {len(CLIPS)} clips: {done}" + (f" | partial (not in the table): {partial}" if partial else ""))
for n, why in SKIPPED.items():
    print(f"SKIPPED {n}: {why}")

# ── earlier results (run 1, EARLIER_DIR) for runs not finished in this session: summary numbers only, marked * ──
EARLIER = {}
_ej, _ec = Path(EARLIER_DIR) / "t12_diag_summary.json", Path(EARLIER_DIR) / "t12_diag_per_item.csv"
if Path(EARLIER_DIR) != OUT_DIR and _ej.exists() and _ec.exists():
    import csv
    _runs = json.load(open(_ej))["summary"]["runs"]
    _rows = list(csv.DictReader(open(_ec)))
    for n, s in _runs.items():
        if n in RUNS and n not in done:
            r = [x for x in _rows if x["run"] == n]
            EARLIER[n] = dict(s, per_clip={x["clip"]: float(x["JF"]) for x in r},
                              cos=np.mean([float(x["cos_2d"]) for x in r]), ic=np.mean([float(x["ic_up"]) for x in r]))
    print(f"earlier results from {EARLIER_DIR} (marked *): {list(EARLIER)}")

# ── table ──
SUM = {n: dict(EARLIER[n]) for n in EARLIER}
print(f"\n{'run':<9} {'what':<36} {'J&F':>6} {'J':>6} {'F':>6} {'Δ vs 2D':>8} {'lost obj-frames':>16}  {'cos(3D,2D)':>10} {'in-cons':>7}   "
      + "  ".join(f"{c:>9}" for c in CLIPS))
for n in done:
    SUM[n] = summary_line(n)[0]
for n in [m for m in RUNS if m in SUM]:                                                  # RUNS order; earlier runs marked *
    if n in done:
        s, lost, pres = summary_line(n)
        cos = np.mean([np.mean(DIAG[n][c]["cos_2d"]) for c in CLIPS]); ic = np.mean([np.mean(DIAG[n][c]["ic_up"]) for c in CLIPS])
        pc = {c: s["per_clip"][c]["JF"] for c in CLIPS}
    else:
        s = EARLIER[n]; lost, pres, cos, ic, pc = s["lost"], s["present"], s["cos"], s["ic"], s["per_clip"]
    dlt = s["J&F-Mean"] - SUM[REF_OF[n]]["J&F-Mean"] if REF_OF[n] in SUM else float("nan")
    print(f"{n + ('*' if n in EARLIER else ''):<9} {DESC[n]:<36} {s['J&F-Mean']:>6.3f} {s['J-Mean']:>6.3f} {s['F-Mean']:>6.3f} {dlt:>+8.3f} {lost:>6}/{pres:<5}({100 * lost / max(pres, 1):4.1f}%)"
          f"  {(f'{cos:.4f}' if RUNS[n]['cond'] == 'anyup3d' else '—'):>10} {ic:>7.4f}   " + "  ".join(f"{pc[c]:>9.3f}" for c in CLIPS))
print("J&F = official, object-weighted over the 3 clips; per-clip columns = clip mean over its objects. in-cons = output avg-pooled to the "
      "LR grid vs the LR features (AnyUp2D on the same backbone for comparison:",
      ", ".join(f"{b} {np.mean([np.mean(DIAG[n][c]['ic_2d']) for c in CLIPS]):.4f}" for b, n in
                (("DINOv2", "ref2d"), ("VideoMAE 224", "v_2d"), ("VideoMAE 448", "v448_2d")) if n in done) + ")"
      + ("\n* = earlier run (EARLIER_DIR), not re-run in this session" if EARLIER else ""))

# ── the comparisons that answer the T12 questions ──
def dJF(a, b):
    return f"{SUM[a]['J&F-Mean'] - SUM[b]['J&F-Mean']:+.3f}" if a in SUM and b in SUM else "n/a (run missing)"
print("\nReading (Δ J&F):")
print(f"  t1 − ref2d   = {dJF('t1', 'ref2d'):>8}   AnyUp3D with no temporal partner vs AnyUp2D (≈ 0 → the temporal part is what hurts)")
print(f"  t1 − t8      = {dJF('t1', 't8'):>8}   gain from removing all temporal context")
print(f"  t2 − t8      = {dJF('t2', 't8'):>8}   training-like context (2) vs 8 frames per call")
print(f"  t8_wt0 − t8  = {dJF('t8_wt0', 't8'):>8}   temporal attention off (convs + time-RoPE stay)")
print(f"  v_3d − v_2d  = {dJF('v_3d', 'v_2d'):>8}   AnyUp3D vs AnyUp2D on AnyUp3D's own training backbone (224, 14² tokens)")
print(f"  v448_2d − v_2d     = {dJF('v448_2d', 'v_2d'):>8}   VideoMAE 448 vs 224 with AnyUp2D (does the higher input help the backbone?)")
print(f"  v448_3d − v448_2d  = {dJF('v448_3d', 'v448_2d'):>8}   AnyUp3D vs AnyUp2D on VideoMAE at a usable resolution  ← main question")
print(f"  v448_t1 − v448_3d  = {dJF('v448_t1', 'v448_3d'):>8}   removing all temporal context (VideoMAE 448)")
print(f"  v448_wt0 − v448_3d = {dJF('v448_wt0', 'v448_3d'):>8}   temporal attention off (VideoMAE 448)")
print(f"  v448_3d − t8       = {dJF('v448_3d', 't8'):>8}   AnyUp3D on its own backbone vs on DINOv2 (both ≈ 30² tokens → 112²)")
if "RES_CHECK" in globals():
    print("VideoMAE 448 feature check (pooled 448 vs 224):", {b: round(float(np.mean(list(v.values()))), 3) for b, v in RES_CHECK.items()})

# ── figure 1: J&F per run + lost object-frames ──
fig, ax = plt.subplots(figsize=(1.25 * max(len(done), 4) + 2, 4.4))
x = np.arange(len(done))
bars = ax.bar(x, [SUM[n]["J&F-Mean"] for n in done], 0.7, color=[COL[n] for n in done])
for i, n in enumerate(done):
    _, lost, pres = summary_line(n)
    ax.text(i, SUM[n]["J&F-Mean"] + 0.01, f"{SUM[n]['J&F-Mean']:.3f}\nlost {100 * lost / max(pres, 1):.1f}%", ha="center", fontsize=7)
for i in range(1, len(done)):                                                           # dotted line between backbone / input groups
    if GROUP[done[i]] != GROUP[done[i - 1]]:
        ax.axvline(i - 0.5, color="k", lw=0.8, ls=":")
ax.set_xticks(x); ax.set_xticklabels([f"{n}\n{DESC[n].replace(' · ', chr(10))}" for n in done], fontsize=7)
ax.set_ylim(0, 1.08); ax.set_ylabel("official J&F (3 clips)"); ax.grid(axis="y", alpha=0.3)
ax.set_title("T12: AnyUp3D variants vs AnyUp2D, this session (groups: backbone + input size; grid 112²)")
plt.tight_layout(); plt.savefig(OUT_DIR / "bars.png", dpi=130, bbox_inches="tight"); plt.show()

# ── figure 2 (per clip): per-frame J, lost frames marked ──
for clip in CLIPS:
    fig, ax = plt.subplots(figsize=(11, 3.4))
    for n in done:
        J = RES[n][clip]["J"].mean(0)                                                    # (T-2,) mean over objects, frames 1..T-2
        ax.plot(np.arange(1, len(J) + 1), J, color=COL[n], lw=1.4, ls="--" if RUNS[n]["cond"] == "anyup2d" else "-",
                label=f"{n} ({clip_mean_jf(RES[n][clip]):.3f})")
        lf = np.where(LOST[n][clip][1][1:-1].any(1))[0] + 1                              # frames with ≥ 1 lost object
        ax.plot(lf, np.full(len(lf), -0.03 - 0.025 * done.index(n)), "|", color=COL[n], ms=6)
    if any(RUNS[n]["bb"] == "dinov2" and RUNS[n]["cond"] == "anyup3d" for n in done):
        for b in range(BB_CHUNK, NFR[clip], BB_CHUNK):
            ax.axvline(b, color="0.85", lw=0.8, zorder=0)
    ax.set_ylim(-0.03 - 0.025 * len(done), 1.02); ax.set_xlabel("frame"); ax.set_ylabel("J (mean over objects)")
    ax.set_title(f"{clip}: per-frame J (dashed = AnyUp2D; ticks below 0 = frames where that run lost an object; grey lines = "
                 f"every {BB_CHUNK} frames)", fontsize=9)
    ax.legend(fontsize=7, loc="upper left", bbox_to_anchor=(1.01, 1)); ax.grid(alpha=0.25)
    plt.tight_layout(); plt.savefig(OUT_DIR / f"per_frame_{clip}.png", dpi=130, bbox_inches="tight"); plt.show()

# ── figure 3 (per clip): feature diagnostics ──
ups = [n for n in done if RUNS[n]["cond"] == "anyup3d"]
for clip in CLIPS:
    fig, axes = plt.subplots(1, 3, figsize=(16, 3.6))
    for n in ups:
        d = DIAG[n][clip]
        axes[0].plot(d["frame"], d["cos_2d"], color=COL[n], lw=1.2, label=n)
        pos = np.array(d["pos"]); c2 = np.array(d["cos_2d"])
        ps = sorted(set(pos))
        axes[2].plot(ps, [c2[pos == p].mean() for p in ps], "-o", ms=4, color=COL[n], label=n)
    for n in done:
        d = DIAG[n][clip]
        axes[1].plot(d["frame"], d["ic_up"], color=COL[n], lw=1.2, ls="--" if RUNS[n]["cond"] == "anyup2d" else "-", label=n)
    axes[0].set_title("cos(AnyUp3D, AnyUp2D), same backbone chunk", fontsize=9); axes[0].set_xlabel("frame")
    axes[1].set_title("input consistency: output pooled to LR vs LR features", fontsize=9); axes[1].set_xlabel("frame")
    axes[2].set_title("cos(AnyUp3D, AnyUp2D) by position in the AnyUp3D call", fontsize=9); axes[2].set_xlabel("position in call")
    for a in axes:
        a.grid(alpha=0.3); a.legend(fontsize=7)
    fig.suptitle(f"{clip}: feature diagnostics", fontsize=11)
    plt.tight_layout(); plt.savefig(OUT_DIR / f"diag_{clip}.png", dpi=130, bbox_inches="tight"); plt.show()


def frame_and_gt(clip, t, hw=None):
    img = np.array(Image.open(davis.frame_paths(ROOT, clip)[t]).convert("RGB"))
    hw = hw or (SHOW_H, int(round(img.shape[1] * SHOW_H / img.shape[0])))
    gt = davis.load_annotation(davis.ann_paths(ROOT, clip)[t])
    return viz.resize_rgb(img, hw), davis.resize_labels(gt, hw), hw


# ── figure 4 (per clip): PCA of the features at the middle frame (one joint PCA per backbone) ──
for clip in CLIPS:
    t = PCA_FRAME[clip]
    groups = [[n for n in done if GROUP[n] == gk and PCA_FEAT[n].get(clip) is not None] for gk in dict.fromkeys(GROUP[m] for m in done)]
    groups = [g for g in groups if g]
    if not groups:
        continue
    ncol = 2 + max(len(g) for g in groups)
    fig, axes = plt.subplots(len(groups), ncol, figsize=(2.6 * ncol, 2.9 * len(groups)), squeeze=False)
    img, _, _ = frame_and_gt(clip, t, hw=GRID)
    for r, g in enumerate(groups):
        feats = {"LR": PCA_FEAT[g[0]][clip]["lr"].float()[None]}                         # (1, C, h, w), same backbone for the whole row
        feats.update({n: PCA_FEAT[n][clip]["up"].float()[None] for n in g})              # (1, C, H, W)
        pca = viz.FeaturePCA(seed=SEED).fit({k: viz.FeaturePCA.sample_tokens(v, PCA_SAMPLE, SEED) for k, v in feats.items()})
        panels = [(img, f"frame {t}")] + [(viz.resize_rgb(pca.rgb(v, k)[0], GRID, nearest=True),
                                           k if k == "LR" else f"{k} ({clip_mean_jf(RES[k][clip]):.3f})") for k, v in feats.items()]
        for a in axes[r]:
            a.axis("off")
        for a, (p, cap) in zip(axes[r], panels):
            a.imshow(p); a.set_title(cap, fontsize=8)
        axes[r][0].set_ylabel(" ".join(GROUP[g[0]]))
    fig.suptitle(f"{clip}: feature PCA at frame {t} (joint basis per backbone row, fixed colour scale; LR = raw backbone grid)", fontsize=10)
    plt.tight_layout(); plt.savefig(OUT_DIR / f"pca_{clip}.png", dpi=120, bbox_inches="tight"); plt.show()

# ── figure 5 (per clip): masks at SHOW_FRACS + the first frame where t8 loses an object ──
for clip in CLIPS:
    T = NFR[clip]
    ts = {min(T - 2, max(1, int(round(f * (T - 1))))) for f in SHOW_FRACS}
    first_lost = None
    focus = next((n for n in ("t8", "v448_3d", "v_3d") if n in done), None)                 # AnyUp3D run whose first loss is shown
    if focus:
        lf = np.where(LOST[focus][clip][1][1:-1].any(1))[0]
        first_lost = int(lf[0]) + 1 if len(lf) else None
        if first_lost is not None:
            ts.add(first_lost)
    ts = sorted(ts)
    fig, axes = plt.subplots(len(ts), 1 + len(done), figsize=(2.7 * (1 + len(done)), 1.75 * len(ts) + 0.8), squeeze=False)
    for r, t in enumerate(ts):
        img, gt_s, hw = frame_and_gt(clip, t)
        panels = [(viz.overlay(img, gt_s), f"GT frame {t}" + (f" ({focus} first loss)" if t == first_lost else ""))]
        panels += [(viz.overlay(img, davis.resize_labels(PRED[n][clip][t], hw)), f"{n} ({clip_mean_jf(RES[n][clip]):.3f})") for n in done]
        for a, (p, cap) in zip(axes[r], panels):
            a.imshow(p); a.set_xticks([]); a.set_yticks([]); a.set_title(cap, fontsize=7)
    fig.suptitle(f"{clip}: propagated masks (run, clip J&F)", fontsize=10)
    plt.tight_layout(); plt.savefig(OUT_DIR / f"masks_{clip}.png", dpi=120, bbox_inches="tight"); plt.show()

# ── videos (per clip): one row per backbone + input group; first panel GT, then the frame with the group name ──
from IPython.display import Video, display
for clip in CLIPS:
    rows_n = [[n for n in done if GROUP[n] == gk] for gk in dict.fromkeys(GROUP[m] for m in done)]
    frames = []
    for t in range(NFR[clip]):
        img, gt_s, hw = frame_and_gt(clip, t)
        rows = []
        for r, g in enumerate(rows_n):
            first = (viz.label_strip(viz.overlay(img, gt_s), f"GT  frame {t}/{NFR[clip] - 1}") if r == 0
                     else viz.label_strip(img, " ".join(GROUP[g[0]]) + " ->"))
            rows.append([first] + [viz.label_strip(viz.overlay(img, davis.resize_labels(PRED[n][clip][t], hw)),
                                                   f"{n} {clip_mean_jf(RES[n][clip]):.3f}") for n in g])
        frames.append(viz.grid(rows))
    mp4 = viz.write_mp4(frames, OUT_DIR / f"{clip}.mp4", VIDEO_FPS)
    viz.write_gif(frames, OUT_DIR / f"{clip}.gif", VIDEO_FPS)
    print("video:", mp4)
    try:
        display(Video(str(mp4), embed=True, width=min(1400, frames[0].shape[1])))
    except Exception as e:   # display only
        print("  (inline preview failed:", e, ")")

# ── save ──
rows = []
for n in done:
    for clip in CLIPS:
        r, d = RES[n][clip], DIAG[n][clip]
        pr, lo = LOST[n][clip]
        rows.append(dict(run=n, **RUNS[n], clip=clip, JF=clip_mean_jf(r), J=float(np.mean(r["J_M"])), F=float(np.mean(r["F_M"])),
                         n_objects=r["n_objects"], lost=int(lo[1:-1].sum()), present=int(pr[1:-1].sum()),
                         cos_2d=float(np.mean(d["cos_2d"])), ic_up=float(np.mean(d["ic_up"])), ic_2d=float(np.mean(d["ic_2d"]))))
summary = dict(runs={n: dict(**{m: SUM[n][m] for m in ("J&F-Mean", "J-Mean", "F-Mean", "n_clips", "n_objects")},
                             lost=summary_line(n)[1], present=summary_line(n)[2], **RUNINFO.get(n, {})) for n in done},
               per_frame_diag={n: DIAG[n] for n in done},
               per_frame_J={n: {c: RES[n][c]["J"].tolist() for c in CLIPS} for n in done},
               skipped=SKIPPED, ckpt=CKPT_INFO, res_check=globals().get("RES_CHECK"))
config = dict(CLIPS=CLIPS, DINO_INPUT=DINO_INPUT, GRID=GRID, BB_CHUNK=BB_CHUNK, VMAE_CHUNK=VMAE_CHUNK, VMAE_HI=VMAE_HI, Q_CHUNK_3D=Q_CHUNK_3D, RUNS=RUNS,
              RUN_FLAGS=dict(RUN_REF=RUN_REF, RUN_T=RUN_T, RUN_V224=RUN_V224, RUN_V448=RUN_V448), RUN_TAG=RUN_TAG,
              TAU=TAU, TOP_K=TOP_K, N_LAST=N_LAST, Q_CHUNK_PROP=Q_CHUNK_PROP, SEED=SEED, CKPT_ANYUP3D=CKPT_ANYUP3D, gpu=GPU_NAME,
              repo_branch=REPO_BRANCH)
csv_p, json_p = metrics.save_results(OUT_DIR, rows, summary, config, name="t12_diag")
print("saved:", csv_p, json_p, "+ figures and videos in", OUT_DIR)